# Big-O & Complexity Analysis

The shared vocabulary for *how an algorithm scales* — the foundation every other notebook in this repo leans on. We'll define Big-O, walk the growth rates you'll keep meeting, show how to read a piece of code's complexity off its loops, and cover the two subtleties that trip people up: **amortized** analysis and **space** complexity.

**Contents**

1. **Why we measure growth**, not seconds
2. **Big-O notation**
3. **The growth rates** you'll meet
4. **Reading complexity off code**
5. **Amortized analysis**
6. **Best / average / worst, and space**
7. **Tractable vs intractable** — P, NP, NP-complete
8. **How to read the cheat-sheets** in these notebooks

## 1. Why we measure growth, not seconds

Wall-clock time depends on your CPU, the language, the input, the weather. To compare *algorithms* we abstract all that away and ask one question: **as the input size n grows, how fast does the work grow?** We count the dominant operations as a function of n and ignore hardware-specific constants.

That's why a "slower" O(n) Python loop still crushes a "faster" O(n²) C loop once n is large enough — **growth rate beats constant factor at scale.**

## 2. Big-O notation

**Big-O** describes an **upper bound** on how an algorithm's cost grows. `O(f(n))` means "for large n, the cost grows no faster than f(n), up to a constant factor." Two rules follow:

- **Drop constant factors:** `3n + 5` is `O(n)`; `n / 2` is `O(n)`. Constants depend on the machine, not the algorithm.
- **Keep only the dominant term:** `n² + 100n + 9` is `O(n²)` — for large n, the `n²` swamps everything else.

(You'll also see **Ω** for a lower bound and **Θ** for a tight bound. When people say "O" casually they usually mean Θ — the rate it *actually* grows at.)

## 3. The growth rates you'll meet

Ordered cheapest to most expensive:

| Big-O | Name | Example | growth, n: 10 → 1000 |
|---|---|---|---|
| O(1) | constant | dict/list index, hash lookup | unchanged |
| O(log n) | logarithmic | binary search, balanced tree | barely grows |
| O(n) | linear | scan a list, linear search | ×100 |
| O(n log n) | linearithmic | the best sorts (Timsort) | ~×150 |
| O(n²) | quadratic | nested loops, bubble sort | ×10,000 |
| O(2ⁿ) | exponential | naive recursive subsets / fib | astronomical |
| O(n!) | factorial | brute-force permutations | hopeless |

The gaps are enormous. Watch how differently they grow:

In [1]:
import math

print(f"{'n':>7}{'log2 n':>9}{'n':>9}{'n log2 n':>12}{'n^2':>13}")
for n in [10, 100, 1000, 10000]:
    print(f"{n:>7}{math.log2(n):>9.1f}{n:>9}{n*math.log2(n):>12.0f}{n**2:>13}")


      n   log2 n        n    n log2 n          n^2
     10      3.3       10          33          100
    100      6.6      100         664        10000
   1000     10.0     1000        9966      1000000
  10000     13.3    10000      132877    100000000


## 4. Reading complexity off code

A few mechanical rules cover most cases:

- **Sequential** steps **add**, so keep the max: O(n) then O(n²) is O(n²).
- **Nested** loops **multiply**: a loop of n inside a loop of n is O(n²).
- **Halving** the problem each step is O(log n) — that's where the log comes from (binary search, tree height).
- A loop that does O(log n) work n times is O(n log n).
- **Recursion**: cost = (number of calls) × (work per call); draw the call tree (the recursion notebook shows naive Fibonacci's exponential one).

The contrast is easy to feel — a linear loop's time roughly **doubles** when n doubles; a quadratic one **quadruples**:

In [2]:
import timeit

def linear(n):
    total = 0
    for i in range(n):
        total += i
    return total

def quadratic(n):
    total = 0
    for i in range(n):
        for j in range(n):
            total += 1
    return total

prev_l = prev_q = None
print(f"{'n':>6}{'linear':>12}{'x':>6}{'quadratic':>14}{'x':>6}")
for n in (500, 1000, 2000):
    tl = timeit.timeit(lambda: linear(n), number=50) / 50
    tq = timeit.timeit(lambda: quadratic(n), number=2) / 2
    rl = "-" if prev_l is None else f"{tl/prev_l:.1f}"
    rq = "-" if prev_q is None else f"{tq/prev_q:.1f}"
    print(f"{n:>6}{tl*1000:>10.3f}ms{rl:>6}{tq*1000:>12.2f}ms{rq:>6}")
    prev_l, prev_q = tl, tq


     n      linear     x     quadratic     x
   500     0.009ms     -        4.51ms     -
  1000     0.020ms   2.2       19.90ms   4.4


  2000     0.037ms   1.8       77.68ms   3.9


## 5. Amortized analysis

Sometimes a single operation is occasionally expensive but **cheap on average over a sequence**. That's **amortized** complexity. The headline example is `list.append`: usually O(1), but when the dynamic array fills it reallocates and copies everything — O(n) for that one append. Those costly resizes are rare enough (capacity grows geometrically) that the *average* over many appends is **O(1) amortized**. (The arrays notebook traces the exact growth.)

You can see it: building a list of N items by appending takes time proportional to N — flat per-item cost — despite the resizes hidden inside:

In [3]:
import timeit

print(f"{'N':>10}{'total':>11}{'per append':>14}")
for N in (250_000, 500_000, 1_000_000):
    def build():
        xs = []
        for i in range(N):
            xs.append(i)
        return xs
    t = timeit.timeit(build, number=1)
    print(f"{N:>10}{t*1000:>9.1f}ms{t/N*1e9:>11.1f} ns")


         N      total    per append
    250000     12.5ms       49.9 ns
    500000     23.9ms       47.9 ns


   1000000     49.4ms       49.4 ns


The per-append nanoseconds stay roughly flat as N grows — that's amortized O(1) in action. (Contrast `list.insert(0, x)`, which is O(n) *every* time — the stacks-and-queues notebook shows that quadratic trap.)

## 6. Best / average / worst, and space

Two more dimensions complete the picture:

- **Best / average / worst case.** Quicksort is O(n log n) *average* but O(n²) *worst* (a pathological pivot); hash lookup is O(1) *average* but O(n) *worst* (everything collides). The cheat-sheets list all three where they differ — assume **worst case** unless told otherwise.
- **Space complexity.** The same idea, for memory. Merge sort needs O(n) scratch space; an in-place sort needs O(1). **Recursion costs space too** — each pending call holds a stack frame, so a depth-d recursion is O(d) space (which is why the recursion notebook cares about depth).

A fast algorithm that needs O(n) scratch space can lose to a slightly slower in-place one when memory is tight — time isn't the only axis.

## 7. Tractable vs intractable — P, NP, NP-complete

Big-O tells you how *one* algorithm scales. A deeper question is whether a *fast algorithm can exist at all*. **Complexity classes** sort problems by that:

- **P** — solvable in **polynomial** time: some algorithm runs in O(n^k) for a fixed k. "Tractable." Sorting, shortest paths, matching — most of this repo lives here.
- **NP** — a proposed solution is **verifiable** in polynomial time, even if *finding* one might not be. Given a candidate, you can check it fast. (P ⊆ NP: if you can solve it fast, you can verify it fast.)
- **NP-complete** — the **hardest** problems in NP: every NP problem **reduces** to them in polynomial time, so a poly-time algorithm for *one* would crack *all* of NP. The classic roster: **SAT** (boolean satisfiability), **subset-sum**, **0/1-knapsack (decision)**, **TSP (decision)**, **graph coloring**, Hamiltonian cycle, clique, vertex cover.
- **NP-hard** — "at least as hard as NP-complete," but **not required to be in NP** (the answer may not be poly-verifiable, or it's an optimization rather than a yes/no). TSP-as-optimization and the halting problem are NP-hard; the halting problem isn't even decidable.

A **polynomial-time reduction** transforms problem A into problem B so that B's answer gives A's — in poly time. If A reduces to B and B is easy, A is easy; contrapositively, if A is hard, B is hard. Reductions are how we *prove* a new problem is NP-complete: reduce a known NP-complete problem to it.

**Whether P = NP is open** — the famous unsolved question. Almost everyone believes P ≠ NP (no fast exact algorithm exists for the NP-complete problems), but nobody has proved it.

```
                NP-hard
              /         \
   NP-complete           (e.g. halting problem —
   (SAT, subset-sum,      NP-hard but not in NP)
    TSP-decision, ...)
        |
  ┌─────┴───────── NP (verifiable in poly time) ─────────┐
  │                                                      │
  │     P (solvable in poly time) — sorting, BFS, ...    │
  └──────────────────────────────────────────────────────┘
        (drawn assuming P ≠ NP — the conjecture)
```

**Why this matters in practice.** Recognizing a problem as NP-complete is a *strategic* result, not a dead end. It tells you to **stop hunting for a guaranteed-fast exact algorithm** — none is known, and probably none exists — and instead reach for one of:

- **Exponential search with pruning** — branch-and-bound, backtracking that cuts dead branches early. Still worst-case exponential, but fast on real instances. (See the **recursion and backtracking** notebook.)
- **Pseudo-polynomial DP** — for subset-sum / 0-1 knapsack, an O(n·W) table is polynomial in the *value* of the numbers but exponential in their *bit-length*; it's a practical win when W is modest. (See the **dynamic programming** notebook.) Splitting the search in half — **meet in the middle** — turns 2^n into 2^(n/2), a huge constant-factor rescue for n in the 30s–40s.
- **Approximation algorithms** — give up exactness for a *provable* ratio (e.g. a 2-approximation for vertex cover, the greedy log-factor for set cover). (See the **greedy** notebook.)
- **Heuristics / metaheuristics** — no guarantee, but good enough fast (simulated annealing, genetic algorithms, LP relaxations).

So the payoff of the theory is concrete: it changes *which tool you grab*.

In [4]:
import random
import timeit

# Subset-sum is NP-complete. The brute force: try ALL 2^n subsets, asking if any
# sums to the target. We force a full scan (impossible target) so each run does the
# worst-case work -- that's what conveys the exponential wall, not a lucky early hit.

def subset_sum_bruteforce(nums, target):
    n = len(nums)
    hits = 0
    for mask in range(1 << n):          # every subset is a bitmask 0 .. 2^n - 1
        s = 0
        m, i = mask, 0
        while m:                        # add the elements whose bit is set
            if m & 1:
                s += nums[i]
            m >>= 1
            i += 1
        if s == target:
            hits += 1
    return hits

rng = random.Random(42)                 # seeded -> reproducible
prev = None
print(f"{'n':>4}{'subsets (2^n)':>16}{'time':>11}{'x prev n':>10}")
for n in range(15, 21):                 # kept small: 2^20 ~ 1M subsets, runs fast
    nums = [rng.randint(1, 100) for _ in range(n)]
    t = timeit.timeit(lambda: subset_sum_bruteforce(nums, -1), number=1)
    ratio = '-' if prev is None else f'{t / prev:.2f}x'
    print(f'{n:>4}{1 << n:>16}{t * 1000:>9.1f}ms{ratio:>10}')
    prev = t

   n   subsets (2^n)       time  x prev n
  15           32768     27.7ms         -
  16           65536     58.9ms     2.12x


  17          131072    111.9ms     1.90x


  18          262144    215.2ms     1.92x


  19          524288    409.8ms     1.90x


  20         1048576    841.7ms     2.05x


Each time n grows by **one**, the work **roughly doubles** (the `x prev n` column hovers near 2x) — because adding one element doubles the number of subsets, 2^n → 2^(n+1). That's the exponential wall: a handful more items and the brute force is hopeless, no faster CPU saves you. This is *exactly* the regime where the NP-completeness of subset-sum tells you to switch tools — to the pseudo-polynomial DP or **meet in the middle** split mentioned above — rather than scaling up the brute force.

## 8. How to read the cheat-sheets in these notebooks

Every notebook ends with a Big-O table. Now you can read them fluently:

- a column per implementation, a row per operation;
- entries are **worst case** unless marked "avg" or "amortized";
- `†` / `‡` footnotes flag the caveats (amortized resizes, collision worst-cases, …);
- "Memory per element" rows are **space**, not time.

With that, the rest of the repo is just filling in *which* structure or algorithm lands in *which* row — and, in the CPython-internals spirit, *why*.